# Comparing seven methods under a common protocol (Protocol 1)

**Goal:** compare seven collaborative filtering methods under exactly the same conditions:

| # | Method | Source |
|---|---|---|
| 1 | Neighborhood (k nearest users) | course, as is |
| 2 | Matrix factorization by gradient descent | course, as is |
| 3 | SVD++: factorization + biases + implicit feedback | Koren (2008), *Factorization Meets the Neighborhood* |
| 4 | Bayesian factorization: factorization + biases, learned by MCMC | "Bayesian MF" row of Rendle, Zhang, Koren (2019) |
| 5 | Bayesian SVD++: 3 and 4 combined | Rendle, Zhang, Koren (2019), *On the Difficulty of Evaluating Baselines* |
| 6 | Bayesian SVD++ *flipped*: method 5 with implicit feedback on the movie side too | "flipped" variant of Rendle, Zhang, Koren (2019), without time |
| 7 | Method 6, averaging several MCMC chains | — |

Methods 3 and 4 each isolate one of the two ingredients of method 5: we see what each brings alone, then together. Method 6 was added **after** seeing the result of method 5; section 9 explains why. Method 7 computes method 6 with several MCMC chains (section 10).

**For each method we obtain:**
1. the best hyperparameters, searched by **Optuna** with **5-fold cross-validation** on the train set;
2. the validation RMSE (mean over the 5 folds);
3. the test RMSE, **measured only once**.

**Protocol 1, identical for the seven methods:**

| Step | What we do | Data |
|---|---|---|
| 1 | Split the train set into 5 folds (the **same** for all methods) | train |
| 2 | For each setting proposed by Optuna: train on 4 folds, measure on the 5th, 5 times; score = mean | train |
| 3 | Keep the setting with the best score (same number of trials for every method) | train |
| 4 | Refit this setting on the **whole** train set, measure **once** on the test set | train → test |

The test set is read only at step 4.

In [ ]:
import os, time, warnings
warnings.filterwarnings("ignore", message="IProgress not found")
os.environ["TQDM_DISABLE"] = "1"                 # no myFM progress bars
import numpy as np
import scipy.sparse as sp
import optuna
import myfm

SEED, N_FOLDS, N_TRIALS = 0, 5, 30
optuna.logging.set_verbosity(optuna.logging.WARNING)

## 1. Data and error measure

Course notation: $R\in\mathbb R^{m\times n}$, one row per **movie** $i$, one column per **user** $u$. The files are stored users × movies, so we transpose them. `NaN` = unknown rating.

The error is the course RMSE, computed only on the cells to predict $T$:
$$\mathrm{RMSE}=\sqrt{\tfrac{1}{|T|}\textstyle\sum_{(i,u)\in T}\big(R_{iu}-\widehat R_{iu}\big)^2}$$
Two conventions, the same for all methods:
- predictions are clipped to the rating range, $[0.5,\,5]$;
- an invalid prediction (a computation that diverged) counts as 0, i.e. as a very bad prediction.

In [ ]:
R_train = np.load("ratings_train.npy").T      # movies × users
R_test  = np.load("ratings_test.npy").T       # loaded here, used only at step 4
m, n = R_train.shape
print(f"{m} movies, {n} users | train: {np.sum(~np.isnan(R_train))} ratings | test: {np.sum(~np.isnan(R_test))} ratings")

def rmse(P, R_true):
    T = ~np.isnan(R_true)
    p = np.clip(np.nan_to_num(P[T], nan=0.0, posinf=5, neginf=0.5), 0.5, 5)
    return float(np.sqrt(np.mean((p - R_true[T]) ** 2)))

4980 movies, 610 users | train: 31598 ratings | test: 31598 ratings


## 2. The 5 cross-validation folds

Each train rating is randomly assigned to one of the 5 folds. For fold $f$: we **train** on the ratings of the other 4 folds and **measure** on those of fold $f$.

**One constraint:** every movie and every user must keep at least one rating in the training part, otherwise we would ask to predict a movie never seen. So for each movie and each user we reserve one random rating, which **always** stays in training and is never measured.

The folds are built **once**, with a fixed seed: the seven methods are evaluated on exactly the same cells.

In [ ]:
rng = np.random.default_rng(SEED)
ii, uu = np.where(~np.isnan(R_train))
fold = rng.integers(0, N_FOLDS, len(ii))
for idx in (ii, uu):                                   # one reserved rating per movie and per user
    for e in np.unique(idx):
        fold[rng.choice(np.where(idx == e)[0])] = -1

FOLDS = []
for f in range(N_FOLDS):
    fit, val = R_train.copy(), np.full_like(R_train, np.nan)
    fit[ii[fold == f], uu[fold == f]] = np.nan
    val[ii[fold == f], uu[fold == f]] = R_train[ii[fold == f], uu[fold == f]]
    FOLDS.append((fit, val))
print("reserved ratings (always in training):", np.sum(fold == -1))
print("validation ratings per fold:", [int(np.sum(~np.isnan(v))) for _, v in FOLDS])

reserved ratings (always in training): 5529
validation ratings per fold: [5326, 5150, 5278, 5101, 5214]


## 3. The common procedure

Every method is written in the same form: `predict(R, R_targets, **hp)` learns from the known ratings of `R` and returns a matrix of predictions (at least on the cells of `R_targets`).

The function `run` applies Protocol 1 to any method:
- **Optuna** proposes `N_TRIALS` settings with the TPE sampler (it proposes new settings where previous ones worked well). The seed is fixed.
- **Score of a setting** = mean RMSE over the 5 folds.
- **Then**: the best setting is refitted on the whole train set and measured **once** on the test set.

In [ ]:
RESULTS = {}

def run(name, predict, space, sampler=None, n_trials=N_TRIALS):
    def objective(trial):
        hp = space(trial)
        scores = [rmse(predict(fit, val, **hp), val) for fit, val in FOLDS]
        trial.set_user_attr("std", float(np.std(scores)))
        return float(np.mean(scores))

    t0 = time.time()
    study = optuna.create_study(direction="minimize", sampler=sampler or optuna.samplers.TPESampler(seed=SEED))
    study.optimize(objective, n_trials=n_trials)
    t_cv = time.time() - t0

    t0 = time.time()
    test = rmse(predict(R_train, R_test, **study.best_params), R_test)      # the only measurement on the test set
    RESULTS[name] = dict(best=study.best_params, val=study.best_value, std=study.best_trial.user_attrs["std"],
                         test=test, t_cv=t_cv, t_fit=time.time() - t0)
    r = RESULTS[name]
    print(f"{name}\n  best hyperparameters: {r['best']}\n"
          f"  validation RMSE (5 folds): {r['val']:.4f} ± {r['std']:.4f}\n  test RMSE: {r['test']:.4f}\n"
          f"  time: search {r['t_cv']/60:.1f} min, final fit {r['t_fit']:.0f} s")

## 4. Method 1 — Neighborhood (course)

**The course algorithm**, to predict $\widehat R_{iu}$:
1. compute $\mathrm{Sim}(u,v)$ for all users $v$;
2. keep at most $k$ nearest neighbors **who rated movie $i$**: $N_k(u;i)$;
3. predict
$$\widehat R_{iu}=\frac{\sum_{v\in N_k(u;i)}\mathrm{Sim}(u,v)\,R_{iv}}{\sum_{v\in N_k(u;i)}\lvert\mathrm{Sim}(u,v)\rvert},$$
and use a **fallback value** if the denominator is zero.

**The two similarities of the course:**
- **Jaccard** on liked movies, $X_u=\{i : R_{iu}\ge 3\}$: $\;\mathrm{Sim}(u,v)=\dfrac{|X_u\cap X_v|}{|X_u\cup X_v|}$;
- **Pearson** on movies rated by both, $M_u\cap M_v$, with $\mu_u$ the mean rating of $u$:
$$\mathrm{Sim}(u,v)=\frac{\sum_{k\in M_u\cap M_v}(r_{ku}-\mu_u)(r_{kv}-\mu_v)}{\sqrt{\sum_{k\in M_u\cap M_v}(r_{ku}-\mu_u)^2}\,\sqrt{\sum_{k\in M_u\cap M_v}(r_{kv}-\mu_v)^2}}$$

**What the course leaves open, and what we chose:**
- the fallback value: $\mu_u$, the user's mean (already defined in the course for Pearson);
- an undefined similarity (no movie in common, or zero denominator) is 0.

**Hyperparameters searched:** $k\in[1,200]$ and the similarity (Jaccard or Pearson).

**Computation:** the similarities between the 610 users fit in a $610\times610$ matrix, computed with matrix products.

In [ ]:
def similarity(R, kind):
    M = ~np.isnan(R)
    if kind == "jaccard":
        X = (np.nan_to_num(R) >= 3).astype(float)                  # X_u = movies rated >= 3
        inter = X.T @ X
        union = X.sum(0)[:, None] + X.sum(0)[None, :] - inter
        return np.divide(inter, union, out=np.zeros_like(inter), where=union > 0)
    Dc = np.where(M, R - np.nanmean(R, 0), 0.0)                    # r_ku - mu_u on rated movies, 0 otherwise
    num = Dc.T @ Dc                                                # sum over M_u ∩ M_v
    A = (Dc ** 2).T @ M                                            # A[u, v] = sum over M_u ∩ M_v of (r_ku - mu_u)^2
    den = np.sqrt(A * A.T)
    return np.divide(num, den, out=np.zeros_like(num), where=den > 0)

def knn(R, R_targets, k, sim):
    S, M, mu = similarity(R, sim), ~np.isnan(R), np.nanmean(R, 0)
    P = np.full(R.shape, np.nan)
    for i, u in zip(*np.where(~np.isnan(R_targets))):
        v = np.where(M[i])[0]                                      # users who rated i
        v = v[np.argsort(-S[u, v])[:k]]                            # the k most similar to u
        s = S[u, v]
        P[i, u] = s @ R[i, v] / np.abs(s).sum() if np.abs(s).sum() > 0 else mu[u]
    return P

run("Neighborhood (course)", knn,
    lambda t: dict(k=t.suggest_int("k", 1, 200, log=True), sim=t.suggest_categorical("sim", ["jaccard", "pearson"])))

Neighborhood (course)
  best hyperparameters: {'k': 144, 'sim': 'jaccard'}
  validation RMSE (5 folds): 0.9944 ± 0.0090
  test RMSE: 1.0060
  time: search 0.5 min, final fit 1 s


**Reading:** Optuna selects **Jaccard with $k=144$**: 0.994 in validation, 1.006 on the test set. This is **worse than predicting each user's mean** (0.945 on the test set in our other notebooks). Two reasons, coming from the data and from the formula:
- **The median movie has only 3 ratings.** The "$k$ nearest neighbors who rated $i$" are therefore almost always *all* those who rated it: with $k=144$, the limit almost never applies. The prediction becomes a similarity-weighted mean of the movie's ratings. It ignores that $u$ rates more harshly or more generously than its neighbors.
- **Pearson is discarded** because its similarities can be negative. The course formula then gives a negative weight to a raw rating (for example $-0.8\times5$), which pushes the prediction outside the rating scale. This formula assumes positive similarities, or centered ratings.

## 5. Method 2 — Matrix factorization (course)

**The course model:** $R\approx I\,U^\top$, with $I\in\mathbb R^{m\times k}$ (one vector per movie) and $U\in\mathbb R^{n\times k}$ (one vector per user).

**The cost, on the observed cells $S$ only** ("Missing values" slide):
$$C(I,U)=\sum_{(i,u)\in S}\Big(R_{iu}-\sum_{s=1}^k I_{is}U_{us}\Big)^2+\lambda\lVert I\rVert_F^2+\mu\lVert U\rVert_F^2$$

**Its derivatives.** Let $E$ be the matrix of errors $R_{iu}-(IU^\top)_{iu}$ on $S$, and 0 elsewhere:
$$\frac{\partial C}{\partial I}=-2\,E\,U+2\lambda I,\qquad \frac{\partial C}{\partial U}=-2\,E^\top I+2\mu U$$

**The algorithm: the course gradient descent**, with a constant step $\eta$ (the same for $I$ and $U$):
$$I_{t+1}=I_t-\eta\,\frac{\partial C}{\partial I}(I_t,U_t),\qquad U_{t+1}=U_t-\eta\,\frac{\partial C}{\partial U}(I_t,U_t)$$
(The course ALS is written for a complete matrix; with missing cells, the course gives the derivatives, hence the choice of gradient descent.)

**What the course leaves open, and what we chose:** the starting point, values drawn from $\mathcal N(0,\,0.1^2)$.

**Hyperparameters searched:** $k\in[1,64]$, $\lambda$ and $\mu\in[10^{-3},10^2]$, $\eta\in[10^{-4},5\cdot10^{-3}]$, number of iterations $\in[100,3000]$. Beyond $\eta\approx3\cdot10^{-3}$ the descent diverges; Optuna notices it and moves away.

**Computation:** errors are computed only on the observed cells (sparse matrix $E$).

In [ ]:
def mf(R, R_targets, k, lam, mu, eta, n_iter):
    rng = np.random.default_rng(SEED)
    ii, uu = np.where(~np.isnan(R)); r = R[ii, uu]
    I = 0.1 * rng.standard_normal((m, k)); U = 0.1 * rng.standard_normal((n, k))
    with np.errstate(all="ignore"):                                # a divergence gives NaN, counted as errors
        for _ in range(n_iter):
            E = sp.csr_matrix((r - np.einsum("ij,ij->i", I[ii], U[uu]), (ii, uu)), shape=(m, n))
            I, U = I - eta * (-2 * E @ U + 2 * lam * I), U - eta * (-2 * E.T @ I + 2 * mu * U)
        return I @ U.T

run("Factorization (course)", mf,
    lambda t: dict(k=t.suggest_int("k", 1, 64, log=True),
                   lam=t.suggest_float("lam", 1e-3, 1e2, log=True), mu=t.suggest_float("mu", 1e-3, 1e2, log=True),
                   eta=t.suggest_float("eta", 1e-4, 5e-3, log=True), n_iter=t.suggest_int("n_iter", 100, 3000, log=True)))

Factorization (course)
  best hyperparameters: {'k': 1, 'lam': 0.014684803764875042, 'mu': 0.9492861281053446, 'eta': 0.0002730449359463328, 'n_iter': 694}
  validation RMSE (5 folds): 0.9049 ± 0.0086
  test RMSE: 0.9123
  time: search 4.0 min, final fit 1 s


**Reading:** Optuna selects **$k=1$**, $\lambda\approx0.015$, $\mu\approx0.95$, $\eta\approx2.7\cdot10^{-4}$, 694 iterations: 0.905 in validation, 0.912 on the test set. This is clearly better than the neighborhood method.
- **$k=1$: a single number per movie and per user**, and the prediction is their product $I_iU_u$. The course model has no mean term: its factors must first produce the average rating level (≈ 3.5). One dimension is enough for that, and each extra dimension adds parameters that overfit poorly rated movies.
- **The step is small and the number of iterations limited**: the descent did not reach the minimum of the cost. Stopping early acts as regularization, which explains why such a small $\lambda$ works.

## 6. Method 3 — SVD++ (Koren 2008)

**The model** (course notation: $q_i=I_i$, $p_u=U_u$):
$$\widehat R_{iu}=\mu+b_i+b_u+\Big\langle q_i\,,\;p_u+\lvert N(u)\rvert^{-1/2}\textstyle\sum_{j\in N(u)}y_j\Big\rangle$$
- $\mu$: mean of all ratings; $b_i$, $b_u$: movie and user biases;
- $\langle q_i,p_u\rangle$: the course factorization;
- $N(u)$: the movies **rated** by $u$ in the input data, whatever the rating. Each movie $j$ has a second vector $y_j$, which says what having rated $j$ reveals about taste.

**Learning as in the paper: stochastic gradient descent.** We go through the ratings one by one (users in random order, then their ratings in random order). For each rating we compute the error $e=R_{iu}-\widehat R_{iu}$, then, with $z_u=p_u+|N(u)|^{-1/2}\sum_{j\in N(u)}y_j$:

| Parameter | Update |
|---|---|
| $b_u$, $b_i$ | $b\leftarrow b+\gamma\,(e-\lambda_b\,b)$ |
| $q_i$ | $q_i\leftarrow q_i+\gamma\,(e\,z_u-\lambda\,q_i)$ |
| $p_u$ | $p_u\leftarrow p_u+\gamma\,(e\,q_i-\lambda\,p_u)$ |
| $y_j$, for all $j\in N(u)$ | $y_j\leftarrow y_j+\gamma\,(e\,|N(u)|^{-1/2}q_i-\lambda\,y_j)$ |

After each pass over all ratings (an **epoch**), the step $\gamma$ is multiplied by 0.9, as in the paper.

**What the paper leaves open, and what we chose:** the starting point, $\mathcal N(0,\,0.1^2)$ for $p_u$, $q_i$, $y_j$ (as for the other methods), and zero biases.

**Hyperparameters searched:** $k\in[2,128]$, $\gamma\in[10^{-3},3\cdot10^{-2}]$, $\lambda_b$ and $\lambda\in[10^{-3},1]$, number of epochs $\in[5,60]$. The values of the paper ($\gamma=0.007$, $\lambda_b=0.005$, $\lambda=0.015$) lie in this space.

**Computation:** the loop over ratings is compiled with `numba` (same algorithm, about 50 times faster than pure Python).

In [ ]:
from numba import njit

@njit
def svdpp_sgd(ptr, items, ratings, mu, bu, bi, P, Q, Y, lr, reg_b, reg, n_epochs):
    np.random.seed(SEED)
    k = P.shape[1]
    for _ in range(n_epochs):
        for u in np.random.permutation(len(bu)):
            its, rs = items[ptr[u]:ptr[u + 1]], ratings[ptr[u]:ptr[u + 1]]      # N(u) and the ratings of u
            c = 1.0 / np.sqrt(max(len(its), 1))
            S = np.zeros(k)
            for j in its: S += Y[j]                                            # sum of the y_j, kept up to date
            for t in np.random.permutation(len(its)):
                i = its[t]
                z = P[u] + c * S
                e = rs[t] - (mu + bu[u] + bi[i] + Q[i] @ z)
                bu[u] += lr * (e - reg_b * bu[u]); bi[i] += lr * (e - reg_b * bi[i])
                qi = Q[i].copy()
                Q[i] += lr * (e * z - reg * Q[i]); P[u] += lr * (e * qi - reg * P[u])
                for j in its:
                    d = lr * (e * c * qi - reg * Y[j]); Y[j] += d; S += d
        lr *= 0.9

def svdpp(R, R_targets, k, lr, reg_b, reg, n_epochs):
    rng = np.random.default_rng(SEED)
    Nu = sp.csc_matrix(~np.isnan(R))                                     # column u: movies rated by u
    ptr, items = Nu.indptr, Nu.indices
    ratings = R[items, np.repeat(np.arange(n), np.diff(ptr))]
    mu, bu, bi = float(np.nanmean(R)), np.zeros(n), np.zeros(m)
    P, Q, Y = (0.1 * rng.standard_normal((s, k)) for s in (n, m, m))
    with np.errstate(all="ignore"):
        svdpp_sgd(ptr, items, ratings, mu, bu, bi, P, Q, Y, lr, reg_b, reg, n_epochs)
        Z = P + sp.diags(1 / np.sqrt(np.maximum(np.diff(ptr), 1))) @ Nu.T @ Y    # z_u for each user
        return mu + bi[:, None] + bu[None, :] + Q @ Z.T

run("SVD++ (Koren 2008)", svdpp,
    lambda t: dict(k=t.suggest_int("k", 2, 128, log=True), lr=t.suggest_float("lr", 1e-3, 3e-2, log=True),
                   reg_b=t.suggest_float("reg_b", 1e-3, 1, log=True), reg=t.suggest_float("reg", 1e-3, 1, log=True),
                   n_epochs=t.suggest_int("n_epochs", 5, 60)))

SVD++ (Koren 2008)
  best hyperparameters: {'k': 5, 'lr': 0.026036270993428392, 'reg_b': 0.10947980755691644, 'reg': 0.19484181194855843, 'n_epochs': 49}
  validation RMSE (5 folds): 0.8712 ± 0.0107
  test RMSE: 0.8767
  time: search 38.0 min, final fit 25 s


**Reading:** Optuna selects **$k=5$, $\gamma\approx0.026$, $\lambda_b\approx0.11$, $\lambda\approx0.19$, 49 epochs**: 0.871 in validation, 0.877 on the test set.
- This is **0.036 better than the course factorization** (0.912). The gain comes from the biases and the implicit feedback, which the course factorization does not have.
- **The settings are far from those of the paper**: about 10 times stronger regularization ($\lambda\approx0.19$ vs 0.015) and only 5 dimensions. The paper was tuned on Netflix, 3,000 times more ratings. Here the median movie has only 3 ratings: strong regularization is needed not to learn them by heart.
- The gap between validation and test (0.0055) is a bit larger than for the other methods.

## 7. Method 4 — Bayesian factorization

**The model:** the course factorization, with biases:
$$\widehat R_{iu}=w_0+w_u+w_i+\langle v_u,v_i\rangle$$

**Bayesian learning.** Instead of looking for *one* set of parameters, we **draw** one at each iteration by Gibbs sampling (MCMC), and we **average** the predictions of all draws. The regularization strengths are not hyperparameters: they are drawn too, from the data, one per group of variables (users, movies).

This is the "Bayesian MF" row of the paper by Rendle, Zhang and Koren (2019), learned with the same software and the same settings as method 5. **The code is that of method 5, without the $N(u)$ block.**

**Hyperparameters searched:** the same as for method 5, rank among 16, 32, 64, 128 and number of iterations in $[64,512]$.

In [ ]:
def bayes_mf(R, R_targets, rank, n_iter):
    user_vars, item_vars = sp.identity(n, format="csr"), sp.identity(m, format="csr")    # [ id u ], [ id i ]
    blocks = lambda i, u: [myfm.RelationBlock(u, user_vars), myfm.RelationBlock(i, item_vars)]

    ii, uu = np.where(~np.isnan(R))
    fm = myfm.MyFMRegressor(rank=rank, init_stdev=0.1, random_seed=SEED)
    fm.fit(sp.csr_matrix((len(ii), 0)), R[ii, uu], X_rel=blocks(ii, uu), n_iter=n_iter, group_shapes=[n, m])
    ti, tu = np.where(~np.isnan(R_targets))
    P = np.full(R.shape, np.nan)
    P[ti, tu] = fm.predict(sp.csr_matrix((len(ti), 0)), blocks(ti, tu))
    return P

run("Bayesian MF", bayes_mf,
    lambda t: dict(rank=t.suggest_categorical("rank", [16, 32, 64, 128]), n_iter=t.suggest_int("n_iter", 64, 512, log=True)))

Bayesian MF
  best hyperparameters: {'rank': 16, 'n_iter': 128}
  validation RMSE (5 folds): 0.8636 ± 0.0110
  test RMSE: 0.8667
  time: search 37.2 min, final fit 5 s


**Reading:** Optuna selects **rank 16 and 128 iterations**: 0.864 in validation, **0.867 on the test set**.
- This is **0.010 better than SVD++, without implicit feedback.** On this data, learning in a Bayesian way (averaging many plausible models and learning the regularization) matters more than adding implicit feedback.
- It is also **very fast**: 5 s for the final fit.

## 8. Method 5 — Bayesian SVD++ (Rendle, Zhang, Koren 2019)

**The model.** It combines methods 3 and 4: the SVD++ model, learned like the Bayesian factorization. Concretely, it is SVD++ written as a *factorization machine*. Each rating is described by three groups of variables (Table 3 of the paper):
- the id of user $u$;
- the id of movie $i$;
- the user's **implicit feedback**: the set $N(u)$ of movies they rated, whatever the rating.

The prediction is then
$$\widehat R_{iu}=w_0+w_u+w_i+\Big\langle v_i\,,\;v_u+\lvert N(u)\rvert^{-1/2}\textstyle\sum_{j\in N(u)}y_j\Big\rangle\;(+\text{interactions between }u\text{ and }N(u)),$$
i.e. the course factorization, plus biases, plus the implicit term of SVD++.

**Bayesian learning.** Instead of looking for *one* set of parameters, we **draw** one at each iteration by Gibbs sampling (MCMC), and we **average** the predictions of all draws. The regularization strengths are not hyperparameters: they are themselves drawn from the data (one per group of variables).

**Faithfulness to the paper:**

| Setting | Paper | Here |
|---|---|---|
| Software | libFM, relational representation | myFM, which reimplements the same sampler with the same relational representation (`RelationBlock`) |
| Variables | $u$, $i$, user implicit feedback | same, one regularization group per variable type |
| Initialization | Gaussian, standard deviation 0.1 | same |
| Dimension | 16, 32, 64, 128 | searched among 16, 32, 64, 128 |
| Iterations | up to 512 | searched in $[64, 512]$ |
| Normalization of $N(u)$ | not specified | $\lvert N(u)\rvert^{-1/2}$, as in SVD++ |
| Content of $N(u)$ | movies rated in train **and** test (without the test ratings) | movies rated in the **input data** only |

On the last point: on the platform we do not know the evaluation cells, so we cannot put them in $N(u)$. We use the strict version, which the paper reports as barely worse.

In [ ]:
def bayes_svdpp(R, R_targets, rank, n_iter):
    M = ~np.isnan(R)
    Nu = sp.csr_matrix(M.T.astype(float))                                   # user -> rated movies
    Nu = sp.diags(1 / np.sqrt(np.maximum(Nu.sum(1).A1, 1))) @ Nu             # normalization |N(u)|^(-1/2)
    user_vars = sp.hstack([sp.identity(n, format="csr"), Nu]).tocsr()       # [ id u | N(u) ]
    item_vars = sp.identity(m, format="csr")                                # [ id i ]
    blocks = lambda i, u: [myfm.RelationBlock(u, user_vars), myfm.RelationBlock(i, item_vars)]

    ii, uu = np.where(M)
    fm = myfm.MyFMRegressor(rank=rank, init_stdev=0.1, random_seed=SEED)
    fm.fit(sp.csr_matrix((len(ii), 0)), R[ii, uu], X_rel=blocks(ii, uu), n_iter=n_iter,
           group_shapes=[n, m, m])                                          # groups: u, N(u), i
    ti, tu = np.where(~np.isnan(R_targets))
    P = np.full(R.shape, np.nan)
    P[ti, tu] = fm.predict(sp.csr_matrix((len(ti), 0)), blocks(ti, tu))
    return P

run("Bayesian SVD++ (paper)", bayes_svdpp,
    lambda t: dict(rank=t.suggest_categorical("rank", [16, 32, 64, 128]), n_iter=t.suggest_int("n_iter", 64, 512, log=True)))

Bayesian SVD++ (paper)
  best hyperparameters: {'rank': 32, 'n_iter': 246}
  validation RMSE (5 folds): 0.8635 ± 0.0098
  test RMSE: 0.8667
  time: search 94.7 min, final fit 29 s


**Reading:** Optuna selects **rank 32 and 246 iterations**: 0.864 in validation, 0.867 on the test set. This is **exactly the score of the Bayesian factorization** (method 4): 0.0001 apart in validation, 0.0000 on the test set, while the standard deviation across folds is 0.01.

**Adding user-side implicit feedback to the Bayesian factorization therefore brings nothing on this data.** This is consistent with its structure:
- users have many ratings (median: 24). Their vector is already well estimated, and the list of their movies adds almost nothing;
- it is the **movies** that have few ratings (median: 3). Implicit feedback helps when added on the movie side: this is the "flipped" variant of the paper, method 6.

The cost is higher than for method 4: 95 minutes of search instead of 37, and 29 s for the final fit instead of 5 s.

## 9. Method 6 — Bayesian SVD++ *flipped*

### Why this method

Method 5 does **not** beat method 4: adding to each user the list of movies they rated brings nothing. The data explain why:

| | Number of train ratings (median) | Consequence |
|---|---|---|
| User | 24 | their vector is already well estimated from their ratings; the list of their movies adds almost nothing |
| Movie | 3 | its vector is very uncertain; any extra information counts |

Implicit feedback should therefore help **on the movie side**: describe a movie also by **the users who rated it**, whatever the rating. This is exactly the "*flipped*" variant of the paper by Rendle, Zhang and Koren (2019), their best variant on MovieLens 10M (0.7485, with time in addition, which our data lack).

### The model

The movie gets the same treatment as the user in SVD++: we add to the movie vector $v_i$ a sum of vectors $x_w$, one per user $w$ in the set $N(i)$ of users who rated $i$:
$$\widehat R_{iu}=w_0+w_u+w_i+\Big\langle\, v_i+\lvert N(i)\rvert^{-1/2}\textstyle\sum_{w\in N(i)}x_w\;,\;\; v_u+\lvert N(u)\rvert^{-1/2}\sum_{j\in N(u)}y_j\Big\rangle\;(+\text{interactions within each block})$$

**The code is that of method 5, with an extra $N(i)$ block on the movie side**, normalized the same way and with its own regularization group. Same search space: rank among 16, 32, 64, 128, iterations in $[64,512]$.

### A methodological precaution

This method was chosen **after** seeing the validation results of methods 4 and 5. This is not a problem as long as the test set stays closed: the choice relies on cross-validation and on the structure of the data, and the test set is measured only once, as for the other methods.

In [ ]:
def bayes_flipped(R, R_targets, rank, n_iter, seed=SEED):
    M = ~np.isnan(R)
    Nu = sp.csr_matrix(M.T.astype(float))                                   # user -> rated movies
    Nu = sp.diags(1 / np.sqrt(np.maximum(Nu.sum(1).A1, 1))) @ Nu             # |N(u)|^(-1/2)
    Ni = sp.csr_matrix(M.astype(float))                                     # movie -> users who rated it
    Ni = sp.diags(1 / np.sqrt(np.maximum(Ni.sum(1).A1, 1))) @ Ni             # |N(i)|^(-1/2)
    user_vars = sp.hstack([sp.identity(n, format="csr"), Nu]).tocsr()       # [ id u | N(u) ]
    item_vars = sp.hstack([sp.identity(m, format="csr"), Ni]).tocsr()       # [ id i | N(i) ]
    blocks = lambda i, u: [myfm.RelationBlock(u, user_vars), myfm.RelationBlock(i, item_vars)]

    ii, uu = np.where(M)
    fm = myfm.MyFMRegressor(rank=rank, init_stdev=0.1, random_seed=seed)
    fm.fit(sp.csr_matrix((len(ii), 0)), R[ii, uu], X_rel=blocks(ii, uu), n_iter=n_iter,
           group_shapes=[n, m, m, n])                                       # groups: u, N(u), i, N(i)
    ti, tu = np.where(~np.isnan(R_targets))
    P = np.full(R.shape, np.nan)
    P[ti, tu] = fm.predict(sp.csr_matrix((len(ti), 0)), blocks(ti, tu))
    return P

run("Bayesian SVD++ flipped", bayes_flipped,
    lambda t: dict(rank=t.suggest_categorical("rank", [16, 32, 64, 128]), n_iter=t.suggest_int("n_iter", 64, 512, log=True)))

Bayesian SVD++ flipped
  best hyperparameters: {'rank': 16, 'n_iter': 145}
  validation RMSE (5 folds): 0.8527 ± 0.0106
  test RMSE: 0.8556
  time: search 81.3 min, final fit 10 s


**Reading:** Optuna selects **rank 16 and 145 iterations**: 0.853 in validation, **0.856 on the test set**. This is the best of methods 1 to 6, with a **gain of 0.011** over methods 4 and 5, both in validation and on the test set.

**The starting hypothesis is confirmed:** implicit feedback only helps where data are missing, i.e. on the movie side. The cost stays low: 10 s for the final fit (81 minutes for the search).

## 10. Method 7 — Method 6 averaging several MCMC chains

### Why this method

The Bayesian prediction is an **average over the draws** of a Markov chain. With 145 iterations, this average keeps some randomness: two chains started with different seeds do not give exactly the same predictions. Lengthening the chain does not help here, since Optuna chose 145 iterations although it could go up to 512: successive draws of one chain are similar. **Running several independent chains** and averaging their predictions explores the posterior better. It is the same method, computed better.

### The setting

We keep the rank and the number of iterations of method 6. **The only hyperparameter is the number of chains $C\in\{1,2,4,8\}$.** Since it has only 4 values, we try them all (full grid, Optuna's `GridSampler`) instead of drawing 30 trials. More chains cost more: $C$ times the time of method 6.

In [ ]:
def bayes_chains(R, R_targets, n_chains):
    hp = RESULTS["Bayesian SVD++ flipped"]["best"]                 # rank and iterations of method 6
    return np.mean([bayes_flipped(R, R_targets, **hp, seed=SEED + c) for c in range(n_chains)], 0)

run("Flipped, several chains", bayes_chains,
    lambda t: dict(n_chains=t.suggest_categorical("n_chains", [1, 2, 4, 8])),
    sampler=optuna.samplers.GridSampler({"n_chains": [1, 2, 4, 8]}), n_trials=4)

Flipped, several chains
  best hyperparameters: {'n_chains': 8}
  validation RMSE (5 folds): 0.8484 ± 0.0110
  test RMSE: 0.8501
  time: search 11.0 min, final fit 85 s


**Reading:** Optuna selects **8 chains**, the largest value tested: 0.848 in validation, **0.850 on the test set**. This is 0.0055 better than method 6 on the test set.
- Fold by fold (computed just below), **the gain over method 6 is positive on all 5 folds**: 0.0043 on average, with a standard deviation of 0.0006. The gain is small but certain.
- **It is the same model, computed better**: method 6 kept some randomness from its MCMC chain.
- **Cost:** 8 times that of method 6, i.e. 85 s for the final fit on the train set. On train+test, full matrix included, it takes 135 s (measured outside the notebook, with the same test RMSE).
- Since 8 is the largest value of the grid, more chains could still help a little, with smaller and smaller gains.

In [ ]:
# Method 7 against method 6, fold by fold (same folds: the difficulty specific to each fold cancels out)
s6 = np.array([rmse(bayes_flipped(fit, val, **RESULTS["Bayesian SVD++ flipped"]["best"]), val) for fit, val in FOLDS])
s7 = np.array([rmse(bayes_chains(fit, val, **RESULTS["Flipped, several chains"]["best"]), val) for fit, val in FOLDS])
g = s6 - s7
print("gain of 7 over 6, fold by fold: " + " ".join(f"{x:+.4f}" for x in g) + f" | mean {g.mean():.4f}, std {g.std():.4f}")

gain of 7 over 6, fold by fold: +0.0033 +0.0041 +0.0050 +0.0047 +0.0043 | mean 0.0043, std 0.0006


## 11. Results

In [ ]:
print(f"{'Method':<28} {'val RMSE (5 folds)':>18} {'test RMSE':>10}   best hyperparameters")
for name, r in RESULTS.items():
    print(f"{name:<28} {r['val']:>11.4f} ± {r['std']:.4f} {r['test']:>10.4f}   {r['best']}")

Method                       val RMSE (5 folds)  test RMSE   best hyperparameters
Neighborhood (course)             0.9944 ± 0.0090     1.0060   {'k': 144, 'sim': 'jaccard'}
Factorization (course)            0.9049 ± 0.0086     0.9123   {'k': 1, 'lam': 0.014684803764875042, 'mu': 0.9492861281053446, 'eta': 0.0002730449359463328, 'n_iter': 694}
SVD++ (Koren 2008)                0.8712 ± 0.0107     0.8767   {'k': 5, 'lr': 0.026036270993428392, 'reg_b': 0.10947980755691644, 'reg': 0.19484181194855843, 'n_epochs': 49}
Bayesian MF                       0.8636 ± 0.0110     0.8667   {'rank': 16, 'n_iter': 128}
Bayesian SVD++ (paper)            0.8635 ± 0.0098     0.8667   {'rank': 32, 'n_iter': 246}
Bayesian SVD++ flipped            0.8527 ± 0.0106     0.8556   {'rank': 16, 'n_iter': 145}
Flipped, several chains           0.8484 ± 0.0110     0.8501   {'n_chains': 8}


**Is the gain of method 6 real?** It is 0.011, about the standard deviation across folds (0.011). But this standard deviation mixes two things: the folds do not all have the same difficulty, and this difficulty affects all methods in the same way. To compare two methods, we therefore look at their difference **fold by fold**.

In [ ]:
# Methods 4 and 6 compared fold by fold, with their best settings (same folds: the difficulty specific to each fold cancels out)
gains = []
for f, (fit, val) in enumerate(FOLDS):
    a = rmse(bayes_mf(fit, val, **RESULTS["Bayesian MF"]["best"]), val)
    b = rmse(bayes_flipped(fit, val, **RESULTS["Bayesian SVD++ flipped"]["best"]), val)
    gains.append(a - b)
    print(f"fold {f}: method 4 {a:.4f} | method 6 {b:.4f} | gain {a - b:+.4f}")
print(f"mean gain {np.mean(gains):.4f}, std of the gain {np.std(gains):.4f}, positive gain on {sum(g > 0 for g in gains)} folds out of 5")

fold 0: method 4 0.8784 | method 6 0.8630 | gain +0.0153


fold 1: method 4 0.8615 | method 6 0.8614 | gain +0.0001


fold 2: method 4 0.8701 | method 6 0.8542 | gain +0.0159


fold 3: method 4 0.8452 | method 6 0.8333 | gain +0.0120


fold 4: method 4 0.8630 | method 6 0.8518 | gain +0.0112
mean gain 0.0109, std of the gain 0.0057, positive gain on 5 folds out of 5


Fold by fold, **method 6 beats method 4 on all 5 folds** (mean gain 0.011, standard deviation of the gain 0.006). The comparison of method 7 against method 6 is in section 10.

**Summary**

| Method | What it adds | Validation RMSE (5 folds) | Test RMSE |
|---|---|---|---|
| 1. Neighborhood (course) | — | 0.994 ± 0.009 | 1.006 |
| 2. Factorization (course) | latent factors | 0.905 ± 0.009 | 0.912 |
| 3. SVD++ (Koren 2008) | biases + user-side implicit feedback | 0.871 ± 0.011 | 0.877 |
| 4. Bayesian MF | biases + Bayesian learning | 0.864 ± 0.011 | 0.867 |
| 5. Bayesian SVD++ (paper) | 3 and 4 combined | 0.864 ± 0.010 | 0.867 |
| 6. Bayesian SVD++ flipped | 5 + movie-side implicit feedback | 0.853 ± 0.011 | 0.856 |
| **7. Flipped, 8 chains** | 6, computed with 8 MCMC chains | **0.848 ± 0.011** | **0.850** |

1. **The ranking is the same in validation and on the test set.** The gaps between neighboring methods are confirmed fold by fold:
   - 6 over 4: 5 folds out of 5;
   - 7 over 6: 5 folds out of 5.
2. **What each ingredient brings:**
   - **biases**: the big jump for factorization (0.912 → 0.877);
   - **Bayesian learning**: 0.010;
   - **user-side implicit feedback**: nothing;
   - **movie-side implicit feedback**: 0.011;
   - **several MCMC chains**: 0.004 to 0.005.
3. **The recommended choice is method 7.** It is a single model, with a gain over method 6 that holds on all 5 folds.
4. **One idea explains the results: information and regularization must go where data are missing.** Here, that is the movies (median of 3 ratings), not the users (24). This is what movie-side implicit feedback does.
5. **The validation RMSE is slightly lower than the test RMSE** for every method. The reserved rating of each movie always stays in training: very poorly rated movies, the hardest ones, therefore rarely appear in validation, while they are in the test set. Since all methods are measured on the same cells, the comparison remains fair.

*The methods were run in several sessions (1, 2 and 5; 3 and 4; 6 and the 4/6 comparison; 7), with the same preparation code and the same seeds. Two functions received an optional argument afterwards: `run` (sampler and number of trials) and `bayes_flipped` (seed). Their default behavior is unchanged. Everything is deterministic: re-running the notebook from start to end gives the same numbers, in about 5 h of computation.*